In [4]:
import os
import shutil

# Root of your combined dataset
BASE = "DroneDatasetCombined"

# Splits to process
SPLITS = ["train", "val", "test"]

# Image extensions supported
IMG_EXTS = [".jpg", ".jpeg", ".png"]

# Outliers folders
OUT_IMG = os.path.join(BASE, "outliers", "images")
OUT_LABEL = os.path.join(BASE, "outliers", "labels")
os.makedirs(OUT_IMG, exist_ok=True)
os.makedirs(OUT_LABEL, exist_ok=True)

def move_outlier(img_path, label_path=None):
    # Move image
    shutil.move(img_path, os.path.join(OUT_IMG, os.path.basename(img_path)))
    # Move label if present
    if label_path and os.path.exists(label_path):
        shutil.move(label_path, os.path.join(OUT_LABEL, os.path.basename(label_path)))

summary = {}

for split in SPLITS:
    img_dir = os.path.join(BASE, "images", split)
    label_dir = os.path.join(BASE, "labels", split)

    removed = 0
    kept = 0

    # Ensure dirs exist
    if not os.path.isdir(img_dir) or not os.path.isdir(label_dir):
        summary[split] = {"removed": removed, "kept": kept}
        continue

    for fname in os.listdir(img_dir):
        # Only process images with supported extensions
        if not any(fname.lower().endswith(ext) for ext in IMG_EXTS):
            continue

        img_path = os.path.join(img_dir, fname)
        base = os.path.splitext(fname)[0]
        label_path = os.path.join(label_dir, base + ".txt")

        # Check label presence and non-empty
        label_missing = not os.path.exists(label_path)
        label_empty = (os.path.exists(label_path) and os.path.getsize(label_path) == 0)

        if label_missing or label_empty:
            move_outlier(img_path, label_path if not label_missing else None)
            removed += 1
        else:
            kept += 1

    summary[split] = {"removed": removed, "kept": kept}

# Print summary
print("✅ Cleanup complete.")
for split, stats in summary.items():
    print(f"{split}: removed={stats['removed']}, kept={stats['kept']}")
print(f"Outliers stored in:\n  images: {OUT_IMG}\n  labels: {OUT_LABEL}")

✅ Cleanup complete.
train: removed=808, kept=3158
val: removed=226, kept=1013
test: removed=111, kept=521
Outliers stored in:
  images: DroneDatasetCombined\outliers\images
  labels: DroneDatasetCombined\outliers\labels
